<a href="https://colab.research.google.com/github/pbandyop/Arena-AI-Safety-courses/blob/main/Evaluating_LLMAgents_InspectAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**In this project I will learn how to evaluate agents in Inspect AI**

In [3]:
# ============================================================
# INSPECT AI + OPENROUTER SETUP FOR GOOGLE COLAB
# ============================================================

# 1. Install packages
!pip install -q -U inspect-ai openai wikipedia


# ============================================================
# 2. Load your OpenRouter API key from Colab Secrets
# ============================================================

import os
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY was not found in Colab Secrets. "
        "Add it under the 🔑 Secrets tab."
    )

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

print("OpenRouter API key available:", bool(os.environ.get("OPENROUTER_API_KEY")))


# ============================================================
# 3. Test Inspect imports
# ============================================================

import inspect_ai

from inspect_ai import Task, eval
from inspect_ai.agent import react
from inspect_ai.dataset import Sample
from inspect_ai.scorer import match
from inspect_ai.tool import tool

print("Inspect AI version:", inspect_ai.__version__)
print("Inspect AI imported successfully ✅")


# ============================================================
# 4. Test that Inspect can resolve the OpenRouter model
# ============================================================

from inspect_ai.model import get_model

model = get_model("openrouter/openai/gpt-4o-mini")

print("Model:", model.name)
print("OpenRouter + Inspect model setup working ✅")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.8/34.8 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.4/94.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.0/83.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.4/731.4 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/

In [4]:
'''
Step 1: Create an agent of your own - I will be using React to create a simple AI agent
Step 2: Create a small dataset to evaluate the agent
Step 3: Run the eval using Inspect AI
'''

# ============================================================
# SIMPLE WIKIPEDIA AGENT + INSPECT AI EVALUATION
# ============================================================

# 1. Install Wikipedia package
!pip install -q wikipedia


# 2. Imports
import wikipedia

from inspect_ai import Task, eval
from inspect_ai.agent import react
from inspect_ai.dataset import Sample
from inspect_ai.scorer import match
from inspect_ai.tool import tool


# ============================================================
# 3. Create a Wikipedia tool
# ============================================================

@tool
def wikipedia_search():
    async def search(query: str) -> str:
        """
        Search Wikipedia for a topic and return the page summary.

        Args:
            query: The topic to search for.
        """
        try:
            page = wikipedia.page(query, auto_suggest=True)

            return (
                f"Title: {page.title}\n\n"
                f"Summary:\n{page.summary}"
            )

        except wikipedia.exceptions.DisambiguationError as e:
            return (
                f"The query '{query}' is ambiguous.\n"
                f"Possible pages: {e.options[:10]}"
            )

        except wikipedia.exceptions.PageError:
            return f"No Wikipedia page found for '{query}'."

        except Exception as e:
            return f"Wikipedia search failed: {e}"

    return search


# ============================================================
# 4. Create the agent
# ============================================================

wiki_agent = react(
    name="wiki_agent",

    description=(
        "An agent that uses Wikipedia to answer factual questions."
    ),

    prompt="""
You are a simple Wikipedia research assistant.

Use the Wikipedia search tool whenever you need factual information.

After receiving information from Wikipedia, answer the user's
question clearly and concisely.

Do not invent facts that are not supported by the Wikipedia result.
""",

    tools=[wikipedia_search()],
)


# ============================================================
# 5. Create a tiny evaluation dataset
# ============================================================

dataset = [
    Sample(
        input=(
            "According to Wikipedia, who developed "
            "the theory of general relativity?"
        ),
        target="Albert Einstein",
    )
]


# ============================================================
# 6. Create the Inspect evaluation task
# ============================================================

task = Task(
    dataset=dataset,
    solver=wiki_agent,
    scorer=match(),
)


# ============================================================
# 7. Run the evaluation
# ============================================================

log = eval(
    task,
    model="openrouter/openai/gpt-4o-mini",
    limit=1,
)


# ============================================================
# 8. Print the result
# ============================================================

print("\n" + "=" * 70)
print("EVALUATION COMPLETE")
print("=" * 70)

print(log)

Output()


EVALUATION COMPLETE
[{
  "version": 2,
  "status": "success",
  "eval": {
    "eval_id": "98WVcbr8DWnxjXYX6pye2H",
    "run_id": "nBnGfaQFYKRVDL42PNp6zt",
    "created": "2026-08-12T20:41:14+00:00",
    "task": "task",
    "task_id": "EN4H4dkJqDJA5TbqM8VDbf",
    "task_version": 0,
    "task_display_name": "task",
    "task_attribs": {},
    "task_args": {},
    "task_args_passed": {},
    "solver_args_passed": {},
    "dataset": {
      "samples": 1,
      "sample_ids": [
        1
      ],
      "shuffled": false
    },
    "model": "openrouter/openai/gpt-4o-mini",
    "model_generate_config": {},
    "model_args": {},
    "config": {
      "limit": 1,
      "epochs": 1,
      "fail_on_error": true,
      "continue_on_fail": false,
      "score_on_error": false,
      "sandbox_cleanup": true,
      "log_samples": true,
      "log_realtime": true,
      "log_images": true,
      "score_display": true
    },
    "packages": {
      "inspect_ai": "0.3.258"
    },
    "scorers": [
     